In [ ]:
# Lets import the needed libraries. There are some quick annotations for what the unusual ones are used for.

import csv
import pandas as pd
import re
import os
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification # An NLP model we use for sentiment analysis, it is pretrained. Details are in README.
from huggingface_hub import login # Our login/API key for hugging face, not strictly needed but helps run smoother.
from dotenv import load_dotenv # Adding secret API keys.
from datetime import datetime
from sklearn.linear_model import LinearRegression # For our regression model.
import matplotlib.pyplot as plt
import sqlite3 # For the purpose of further demonstrating the skills learned in this model, I chose to use SQL here.

# Below we load our API keys from the .env. You will require your own API keys, this is for the purpose of hiding mine.
# In this script we only use a HuggingFace key, althought this is not strictly required for the code to run, it does make it run smoother.

load_dotenv()
token = os.getenv('HF_TOKEN')
login(token)


In [ ]:
# In this cell we quickly load our data. We combine the Guardian News data and the NewsAPI data into one dataframe (news) via concat.

News_api_data = pd.read_csv('NewsAPI_data.csv',encoding='utf-8') # Has some ecoding issues earlier, so just ensuring it is utf-8.
guardian_data = pd.read_csv('guardian_data.csv',encoding='utf-8')
guardian_data = guardian_data.rename(columns={'headline': 'title', 'body_text': 'content'}, inplace=True) # Quickly rename Guardian columns so it will merge properly.
news = pd.concat([guardian_data,News_api_data])

# We also load the financial data as well.

brent = pd.read_csv('Brent_data.csv',encoding='utf-8')
wti = pd.read_csv('WTI_data.csv',encoding='utf-8')
ovx = pd.read_csv('ovx_data.csv',encoding='utf-8')
vix = pd.read_csv('vix_data.csv',encoding='utf-8')


In [ ]:
# This cell runs our sentiment analysis models over our news data.
# We use the ProsusAI/finbert model.

prosus_model= AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert') # Defining the model and tokenizer.
prosus_tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')

# We use batching to speed up the process, if you are on a powerfull computer, you can increase the batch_size to speed up the run time.
nlp1 = pipeline('text-classification', model=prosus_model, tokenizer=prosus_tokenizer, truncation=True, max_length=512,batch_size=32)

# Creating a function to run over the title and content columns, it returns out the sentiment scores.
def sentiment(texts):
    results1 = nlp1([i if isinstance(i, str) and i.strip() else ' ' for i in texts])
    return [ i['score'] if i['label'] == 'positive' else -i['score'] if i['label'] == 'negative' else 0 for i in results1] 

news['title_score_prosus'] = sentiment(news['title'])
news['content_score_prosus'] = sentiment(news['content'])
news['final_score_prosus'] = news[['title_score_prosus','content_score_prosus']].mean(axis=1) # We create an additional column for an average score (of tile and content).

In [ ]:
# Quickly saving the news and sentiment data so we dont have to rerun the above. 
# We load it again in the next cell.

news.to_csv('important_one.csv', index=False, encoding='utf-8')

In [ ]:
# Merging later on throws a date data type error, so we again ensure everything is the same date data type here.

news = pd.read_csv('important_one.csv', encoding='utf-8')
news['date'] = pd.to_datetime(news['date'], format='%Y%m%d')
brent['date'] = pd.to_datetime(brent['date'])
wti['date'] = pd.to_datetime(wti['date'])
ovx['date'] = pd.to_datetime(ovx['date'])
vix['date'] = pd.to_datetime(vix['date'])

In [ ]:
brent['date'].dtype

In [ ]:
# We create new dataframes with the average daily sentiment using the final average score, title score alone, and content score alone.
# We can compare the sentiment between titles and content this way.
# We get the mean, but standard deviation and count will also be usefull

sentiment_by_day_for_avg = pd.DataFrame(news.groupby('date')['final_score_prosus'].agg(['mean','std','count']))
sentiment_by_day_for_title = pd.DataFrame(news.groupby('date')['title_score_prosus'].agg(['mean','std','count']))
sentiment_by_day_for_content = pd.DataFrame(news.groupby('date')['content_score_prosus'].agg(['mean','std','count']))

sentiment_by_day_for_avg.reset_index(inplace=True) # Resetting the index.
sentiment_by_day_for_title.reset_index(inplace=True)
sentiment_by_day_for_content.reset_index(inplace=True)

In [ ]:
fig, ax1 = plt.subplots(figsize=(20, 5))

ax1.plot(sentiment_by_day_for_avg['date'], sentiment_by_day_for_avg['mean'], color='#384358', linewidth=2.5, label='Total Average Per Day')
ax1.plot(sentiment_by_day_for_title['date'], sentiment_by_day_for_title['mean'], color='#BC1139', linewidth=2.5, label='Headline Score Per Day')
ax1.plot(sentiment_by_day_for_content['date'], sentiment_by_day_for_content['mean'], color='#739ab9', linewidth=2.5, label='Content Score Per Day')

ax1.set_ylabel('Sentiment Average', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')
ax1.legend(loc='upper left')
plt.xticks(rotation=45, fontsize=7, fontfamily='monospace')
plt.yticks(fontfamily='monospace')

plt.title('Averaged Sentiment VS Title Only VS Content Only', fontsize=13, fontfamily='monospace')
plt.tight_layout()
plt.show()

In [ ]:
pre_complete1 = pd.merge(sentiment_by_day_for_avg, brent ,on='date', how='inner')
pre_complete2 = pd.merge(pre_complete1, wti,on='date', how='inner')
pre_complete3 = pd.merge(pre_complete2, vix,on='date', how='inner')
pre_complete3 = pre_complete3.rename(columns={'mean': 'sentiment_mean','std': 'sentiment_std','daily_avg_x': 'brent','daily_avg_y': 'wti','daily_avg': 'vix'})
complete = pd.merge(pre_complete3, ovx,on='date', how='inner')
complete = complete.rename(columns={'daily_avg':'ovx'})

In [ ]:
# While this section could be done in python, for the purpose of further demonstrating the skills learned in this model, I chose to use SQL here.

conn = sqlite3.connect('newsdata.db')
complete.to_sql('news_and_data',conn,if_exists='replace',index=False)
conn.commit()


def movingaverage(column,new_column):
    conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                       AVG({column}) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
                       AS {new_column}
                       FROM news_and_data;''')
    conn.execute("DROP TABLE news_and_data;")
    conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
    conn.commit()
    
movingaverage('sentiment_mean','sentiment_mean_moving_average')
movingaverage('ovx','ovx_moving_average')
movingaverage('vix','vix_moving_average')
movingaverage('wti','wti_moving_average')
movingaverage('brent','brent_moving_average')

def invert(column,new_column):
    conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                       -{column} AS {new_column}
                       FROM news_and_data;''')
    conn.execute("DROP TABLE news_and_data;")
    conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
    conn.commit()

invert('sentiment_mean_moving_average','sentiment_mean_moving_average_inversed')

def lag(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        (LAG({column}) OVER (ORDER BY date)) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()

lag('brent','lagged_brent')

def abs_value(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        ABS({column}) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()

abs_value('sentiment_mean','sentiment_mean_abs')

def percentage_change(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        (((({column}) - (LAG({column}) OVER(ORDER BY date))) / (LAG({column}) OVER(ORDER BY date)))*100) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()
     
percentage_change('brent','brent_percent')
percentage_change('wti','wti_precent')

    

In [ ]:
All_needed_data = pd.read_sql('''SELECT * FROM news_and_data''',conn)
All_needed_data['date'] = pd.to_datetime(All_needed_data['date'], format='ISO8601')

In [ ]:
temp = All_needed_data[['wti_precent','sentiment_mean_abs','brent_percent']].dropna()
X = temp[['brent_percent']]
Y = temp['sentiment_mean_abs']

model = LinearRegression()
model.fit(X, Y)

print(model.coef_, model.intercept_)
print(model.score(X, Y))

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.bar(All_needed_data['date'], All_needed_data['sentiment_mean'], color='#2D244D', alpha=0.7, width=0.8, label='Daily Sentiment')
ax1.axhline(0, color='black')
ax1.set_ylabel('Sentiment by day\n(Lower is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')
ax1.set_ylim(-1, 0.3)

ax2 = ax1.twinx()
ax2.plot(All_needed_data['date'], All_needed_data['brent'], color='#FF5B04', linewidth=2.5, label='Brent Crude (USD)')
ax2.plot(All_needed_data['date'], All_needed_data['wti'], color='#C2441C', linewidth=2.5, label='WTI Crude (USD)')
ax2.set_ylabel('Oil Price (USD/barrel)', fontsize=11, fontfamily='monospace')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('News Sentiment vs Brent Crude Price Over the Iran War', fontsize=15, fontweight='bold', fontfamily='monospace')
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(All_needed_data['date'], All_needed_data['sentiment_mean_moving_average_inversed'], color='#BC1139', linewidth=2.5, label='News Sentiment Inversed 3 Day Moving Average')
ax1.set_ylabel('Sentiment Average Inversed as Fear\n(Higher is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')

ax2 = ax1.twinx()
ax2.plot(All_needed_data['date'], All_needed_data['ovx_moving_average'], color='#739ab9', linewidth=2.5, label='Oil Volatility Index (^OVX) 3 Day Moving Average')
ax2.set_ylabel('Oil Volatility Index\n(Higher is more Volatile)', fontsize=11, fontfamily='monospace')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Market Fear vs Oil Volatility Index', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(All_needed_data['date'], All_needed_data['ovx_moving_average'], color='#BC1139', linewidth=2
        , label='OVX moving average (Oil Volatility)')
ax.plot(All_needed_data['date'], All_needed_data['vix_moving_average'], color='#739ab9', linewidth=2
        , label='VIX moving average (Broad Market)')
ax.set_ylabel('Volatility Index')
ax.set_title('Oil Fear (OVX) vs Broad Market Fear (VIX) During Iran War\n''(whe war was an energy shock, not a systemic market panic)',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()


#normalise the 2, this can be an introductory graph

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(All_needed_data['date'], All_needed_data['sentiment_mean_moving_average_inversed'], color='#BC1139', linewidth=2.5, label='News Sentiment Inversed 3 Day Moving Average')
ax1.set_ylabel('Sentiment Average\n(Higher is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')

ax2 = ax1.twinx()
ax2.plot(All_needed_data['date'], All_needed_data['vix_moving_average'], color='#739ab9', linewidth=2.5, label='Market Volatilty Index (^VIX) 3 Day Moving Average')
ax2.set_ylabel('Oil Volatility Index\n(Higher is more Volatile)', fontsize=11, fontfamily='monospace')


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left')

plt.title('Market Fear vs Market Volatility Index', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(All_needed_data['date'], All_needed_data['sentiment_mean'], color='#BC1139', linewidth=2.5, label='News Sentiment Per Day')
ax1.set_ylabel('Sentiment Per Day (Lower is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')

ax2 = ax1.twinx()
ax2.plot(All_needed_data['date'], All_needed_data['lagged_brent'], color='#739ab9', linewidth=2.5, label='Lagged Brent Oil USD')
ax2.set_ylabel('Brent USD', fontsize=11, fontfamily='monospace')


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

plt.title('Sentiment Per Day vs Brent Oil', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()